# AIRA Model Fine-Tuning — QLoRA SFT on Google Colab
### Autonomous Infrastructure Resilience Architecture

This notebook implements the **Phase 4** SFT (Supervised Fine-Tuning) pipeline for **AIRA**. 
It uses standard **HuggingFace TRL + bitsandbytes QLoRA** training on a free-tier Google Colab T4 GPU to fine-tune `google/gemma-4-e4b-it` on our self-generated adversarial trajectory dataset (`sft_dataset.jsonl`).

At the end of training, it exports the fine-tuned LoRA adapter weights.

### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
# Install standard HuggingFace SFT and bitsandbytes dependencies with transformers from source
!pip install git+https://github.com/huggingface/transformers.git
!pip install trl peft accelerate bitsandbytes datasets
!pip install pydantic structlog

### 2. Load Model and Tokenizer (4-bit Quantization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "google/gemma-4-e4b-it"
max_seq_length = 1024         # Set memory-safe sequence limit to cut logits size in half

print(f"[*] Loading model and tokenizer in 4-bit QLoRA for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    attn_implementation="sdpa",   # Force PyTorch Scaled Dot Product Attention for memory efficiency
    device_map={"": 0}           # Load completely on GPU 0 to prevent meta splits and speed up training
)
print("[SUCCESS] 4-bit QLoRA Model loaded successfully on GPU!")

### 3. Configure LoRA Adapters (Rank 64 / Alpha 128)

In [ ]:
from peft import LoraConfig, get_peft_model

# Configure standard PEFT LoRA, targeting the inner .linear child modules inside Gemma 4's custom wrappers
peft_config = LoraConfig(
    r=64,  # LoRA Rank
    lora_alpha=128,  # LoRA Alpha
    target_modules=["q_proj.linear", "k_proj.linear", "v_proj.linear", "o_proj.linear", 
                    "gate_proj.linear", "up_proj.linear", "down_proj.linear"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 4. Load & Format Trajectory Dataset (`sft_dataset.jsonl`)

In [ ]:
import os
from datasets import load_dataset

dataset_path = "/content/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "/kaggle/input/aira-sft-dataset/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "/kaggle/input/sft-dataset/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "sft_dataset.jsonl"

print(f"[*] Loading trajectory dataset from {dataset_path}...")
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True, remove_columns=dataset.column_names)
print(f"[SUCCESS] Dataset loaded! Formatted {len(dataset)} SFT samples. Columns: {dataset.column_names}")

### 5. Training Configuration (TRL SFTTrainer)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        max_seq_length = max_seq_length,       # Pass inside SFTConfig to resolve newest TRL version requirements
        per_device_train_batch_size = 1,       # Reduce batch size to 1 to cut VRAM usage in half
        gradient_accumulation_steps = 8,       # Maintain effective batch size of 8
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = True,
        gradient_checkpointing = True,
        logging_steps = 1,
        optim = "paged_adamw_8bit",            # Paged 8-bit optimizer to save 1.2 GB VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "no"
    ),
)

In [ ]:
trainer_stats = trainer.train()

### 7. Export weights to 4-bit GGUF (for local Ollama deployment)

In [ ]:
# Save the fine-tuned LoRA adapter weights in PyTorch format to bypass Safetensors copy limitations
model.save_pretrained("gemma-4-e4b-aira-lora", safe_serialization=False)
tokenizer.save_pretrained("gemma-4-e4b-aira-lora")

# To deploy locally via Ollama:
# 1. Download your LoRA adapter folder from Colab.
# 2. Use a tool like llama.cpp to merge and quantize the model, or push the adapter directly to Hugging Face.